In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import *
from IPython.display import display, Markdown as IMarkdown
from rich.console import Console
from rich.markdown import Markdown

[16:08:53] cuDF not found: falling back to standalone pandas.

In [2]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

In [3]:
dataset = automed.Dataset(titanic.iloc[:200].copy())
dataset.set_label(['label'])

autom = automed.AutoMed(dataset=dataset, max_workers=1)

In [4]:
print(autom.debug_load())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'RandomSplit', 'name': 'Split date to train and test set', 'description': 'Step description...', 'configuration': {'ratio': {'description': 'Split ratio', 'default': 0.2, 'value': 0.2}, 'random_state': {'description': 'Random state', 'default': 42, 'value': 42}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropNumericalColumn', 'name': 'Drop numerical columns', 'description': 'Drop numerical columns where the proportion of empty rows in the dataset is higher than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.5}}, 'children': []}, {'step': 'ActDropTextualColumn', 'name': 'Drop textual columns', 'description': 'Drop

In [5]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [{
        'step': 'RandomSplit',
        'configuration': {
            'ratio': {
                'value': 0.3
            }
        }},
        {
            'step': 'ActDropNumericalColumn',
            'configuration': {
                'empty_threshold': { 'value': 0.1 },
            }
        },
        {
            'step': 'MetaStep',
            'tag': 'cleaning',
        },
        {
            'step': 'MetaStep',
            'tag': 'features_selection',
        },
        {
            'step': 'MetaStep',
            'tag': 'metric'
        },
        {
            'step': 'MetaExplorerStep',
            'tag': 'learning'
        }]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'RandomSplit', 'name': 'Split date to train and test set', 'description': 'Step description...', 'configuration': {'ratio': {'description': 'Split ratio', 'default': 0.2, 'value': 0.3}, 'random_state': {'description': 'Random state', 'default': 42, 'value': 42}}, 'children': []}, {'step': 'ActDropNumericalColumn', 'name': 'Drop numerical columns', 'description': 'Drop numerical columns where the proportion of empty rows in the dataset is higher than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.1}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropNumericalColumn', 'name': 'Drop numerical columns', 'description': 'Drop 

In [6]:
results = autom.run(callback=lambda step: print(step, id(step), step.parents_steps))
[ (r.pipeline.model, r.evaluate()) for r in results if r.pipeline.model is not None ]

Output()

           running step: MetaOrderedStep (steps=MetaExplorerStep,MetaStep,ActDropNumericalColumn,RandomSplit)

Split date to train and test set 1552175494352 [1552175493712]

Drop numerical columns 1552175493648 [1552175493712]

           running step: MetaStep                                                                                  
           (steps=ActOnehot,ActDropTextualColumn,ActSplitDate,ActMeanColumn,ActDropNumericalColumn,ActDropDateColum
           n)

...

Learn : SVM Classification 1552175532816 [1552175532496, 1552175493712]

Learn : XGBoost 1552175538832 [1552175532496, 1552175493712]

MetaExplorerStep 1552175532496 [1552175493712]

MetaStep 1552175493712 []

[(<automed.actionables.learning.act_logistic_regression.ActLogisticRegression at 0x16964df0e10>,
  {'accuracy': 0.8,
   'balanced_accuracy': 0.7689345314505777,
   'classification_error': 0.23106546854942234,
   'f1_score': 0.8536585365853658,
   'precision': 0.8536585365853658,
   'recall': 0.8536585365853658,
   'specificity': 0.8536585365853658}),
 (<automed.actionables.learning.act_randomforest.ActRandomForest at 0x16964df1590>,
  {'accuracy': 0.8,
   'balanced_accuracy': 0.754813863928113,
   'classification_error': 0.245186136071887,
   'f1_score': 0.8571428571428572,
   'precision': 0.8372093023255814,
   'recall': 0.8372093023255814,
   'specificity': 0.8780487804878049}),
 (<automed.actionables.learning.act_xgboost.ActXGBoost at 0x16964df0b50>,
  {'accuracy': 0.7333333333333333,
   'balanced_accuracy': 0.6777920410783055,
   'classification_error': 0.3222079589216945,
   'f1_score': 0.8095238095238095,
   'precision': 0.7906976744186046,
   'recall': 0.7906976744186046,
   'sp

In [7]:
print(results[0].pipeline.model)
print(results[0].pipeline.steps)

console = Console()

for step in results[0].pipeline.explanations:
    md = step.to_markdown()
    if md:
        # console.print(Markdown(md))
        display(IMarkdown(md))

pm = results[0].pipeline.pickle()

Learn : Logistic Regression Classifier
[('Drop numerical columns', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x0000016964DE7210>), ('Fill missing values with mean', <automed.actionables.cleaning.act_mean_column.ActMeanColumn object at 0x0000016964DE7F50>), ('One hot encoding categorical features', <automed.actionables.cleaning.act_onehot.ActOnehot object at 0x0000016964DE7A90>), ('Transform string column to date', <automed.actionables.cleaning.act_split_date.ActSplitDate object at 0x0000016964DE7E10>), ('Drop numerical columns', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x0000016964DE7650>), ('Drop textual columns', <automed.actionables.cleaning.act_drop_textual_column.ActDropTextualColumn object at 0x0000016964DE7810>), ('Drop date columns', <automed.actionables.cleaning.act_drop_date_column.ActDropDateColumn object at 0x0000016964DE7BD0>), ('Remove High Correlated Column', <automed.actionabl


## Drop numerical columns
**Drop numerical columns where the proportion of empty rows in the dataset is higher than 0.1.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **empty_threshold** | Column with more or equal proportion of empty row will dropped. 1 will drop all columns | 0.1 |



### Processings
 - Dropped column **`Age`** because **27** values out of **140** (**19.29%**) are empty.

            


## One hot encoding categorical features
**Step description...**




### Processings
 - Encoded categorical column **`Sex`** into **2** new columns.
 - Encoded categorical column **`Embarked`** into **4** new columns.

            


## Drop textual columns
**Drop textual columns.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **ratio** | Description of the parameter's role | 0.8 |
| **random_state** | Description of the parameter's role | 12 |



### Processings
 - Dropped column **`Name`**.
 - Dropped column **`Ticket`**.
 - Dropped column **`Cabin`**.

            


## Remove High Correlated Column
**Remove columns which correlation with other columns is higher than 0.9.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **threshold** | If two columns is correlated over this value, only one will be kept | 0.9 |



### Processings
 - Dropped column **`Sex_male`** because it was too correlated with **`Sex_female`**.

            

In [8]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels[o+i] for i, r in enumerate(m.run(predict_df)) ]) / n

NameError: name 'pm' is not defined

In [ ]:
final_boss_dataset = Dataset(titanic.copy())
final_boss_dataset.set_label('label')

final_boss_automed = AutoMed(final_boss_dataset, max_workers=2)
final_boss_automed.debug_load()
final_boss_results = final_boss_automed.run()

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]